In [1]:
import os
import pandas as pd
import numpy as np
import scipy.signal as signal
from tqdm import tqdm
from scipy.io import loadmat

In [2]:
MI_dir = r"F:\M143020071\MI\raw data"
H_and_P_data_path = r"F:\M143020071\MI\new_dataname_400_1.mat"
csv_dir = r"F:\M143020071\MI\combined_sknaMI.csv"
save_dir = r'D:\M143020071\MI\raw_data_result\dunwei\ch1\sr500_0.5_50_MI_win10s_step2s_2-7m/'

In [3]:
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"Directory '{save_dir}' created.")
else:
    print(f"Directory '{save_dir}' already exists.")

Directory 'D:\M143020071\MI\raw_data_result\dunwei\ch1\sr500_0.5_50_MI_win10s_step2s_2-7m/' created.


In [4]:
H_and_P_data = loadmat(H_and_P_data_path)['H_and_P']
csv = pd.read_csv(csv_dir)
ID_LABEL = csv[csv["Selected"] == True][['ID Number', 'MI']]
print(ID_LABEL)

     ID Number     MI
1            2   True
2            3   True
6            7   True
9           10   True
10          11   True
..         ...    ...
900        901  False
902        903  False
908        909  False
909        910  False
920        921  False

[400 rows x 2 columns]


In [5]:
original_sr = 10000
ecg_resample_rate = 500
ecg_highcut = 50
ecg_lowcut = 0.5
window_size = 10
window_stride = 2

In [7]:
win_num_dict = {'research_id':[], 'win_num':[], 'label_feature_num':[]}
for i in tqdm(range(len(H_and_P_data)), desc="Processing files"):
    filename = H_and_P_data[i][0][0]
    convert_id = filename.split('.')[0]
    if filename.endswith('.mat'):
        try:
            file_path = os.path.join(MI_dir, filename)
            mi_signal = loadmat(file_path)['data'][:, 0] # 讀取第一個column signal
            highpass_sos = signal.butter(6, ecg_lowcut, btype='highpass', output='sos', fs=original_sr)
            highpass_signal = signal.sosfiltfilt(highpass_sos, mi_signal)
            lowpass_sos = signal.butter(6, ecg_highcut, btype='lowpass', output='sos', fs=original_sr)
            ecg_filtered_signal = signal.sosfiltfilt(lowpass_sos, highpass_signal)
            ecg_signal_resampled = signal.decimate(ecg_filtered_signal, int(original_sr/ecg_resample_rate), ftype='iir', zero_phase=True)
            if ecg_signal_resampled.shape[0] >= 2 * ecg_resample_rate * 60:
                if ecg_signal_resampled.shape[0] < 7 * ecg_resample_rate * 60:
                    original_offset = ecg_signal_resampled.shape[0] - 5 * ecg_resample_rate * 60
                    ecg_signal_segment_5min = ecg_signal_resampled[original_offset : ]
                else:
                    original_offset = 2 * ecg_resample_rate * 60
                    end_offset = 7 * ecg_resample_rate * 60
                    ecg_signal_segment_5min = ecg_signal_resampled[original_offset : end_offset]
            else:
                print(f"File {filename}.csv is too short Skipping.")
                continue
            

            segment_win_ecg_list = []
            for start in range(0, len(ecg_signal_segment_5min) - (window_size * ecg_resample_rate) + 1, (window_stride * ecg_resample_rate)):
                end = start + window_size * ecg_resample_rate
                segment_win_ecg = ecg_signal_segment_5min[start:end]
                segment_win_ecg_list.append(segment_win_ecg)
            

            ecg_signal = np.vstack(segment_win_ecg_list)
            mu = np.mean(ecg_signal, axis=1, keepdims=True)
            sigma = np.std(ecg_signal, axis=1, keepdims=True)
            ecg_signal_nor = (ecg_signal - mu) / (sigma + 1e-8)

            if ID_LABEL[ID_LABEL['ID Number'] == int(filename.split('.')[0])]['MI'].values[0] == False:
                label_arr = np.zeros((ecg_signal_nor.shape[0],1))
            elif ID_LABEL[ID_LABEL['ID Number'] == int(filename.split('.')[0])]['MI'].values[0] == True:
                label_arr = np.ones((ecg_signal_nor.shape[0],1))
            iskna_signal_with_label = np.concatenate((label_arr, ecg_signal_nor), axis=1).astype(np.float32)
            np.save(os.path.join(save_dir, f'{convert_id}.npy'), iskna_signal_with_label)
            
            win_num_dict['research_id'].append(f'{convert_id}')
            win_num_dict['win_num'].append(int(ecg_signal_nor.shape[0]))
            win_num_dict['label_feature_num'].append(iskna_signal_with_label.shape[1])
            
        except Exception as e:
            print(f"Error loading {filename}: {e}")
            continue

win_num_df = pd.DataFrame(win_num_dict)
win_num_df.to_csv(os.path.join(save_dir, 'window_numbers.csv'), index=False)

Processing files: 100%|██████████| 400/400 [04:45<00:00,  1.40it/s]
